# FOLIO Shared Config + Login

This notebook holds the FOLIO connection settings (Okapi URL, tenant, username) and the login logic, so other notebooks don't need to repeat this action. 

**To use if from another notebook,** both notebooks must be in the same folder. Then, include the following  in the first line of your notebook:

%run folio_auth.ipynb

**Running on Google Colab?** See the comments in the cells below -- this notebook auto-detects Colab and switches how it reads your password (Colab Secrets instead of a local environment variable). No changes needed on your end beyond adding a Colab secret named `FOLIO_PASSWORD`.

In [1]:
import requests
import getpass
import os

# --- Colab detection ---
# `google.colab` is a package that only exists inside Google Colab's runtime
# -- it's pre-installed there and nowhere else. So this import succeeding or
# failing is a reliable way to tell "am I running on Colab or on a normal
# local Jupyter install?" We use IN_COLAB below to decide how to fetch your
# FOLIO password, since Colab has no persistent shell to set an environment
# variable in the way local Jupyter does.
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# clear out any previous session data
session=''
token=''

## Configuration

Edit the values below for your FOLIO tenant. Any notebook that first calls this notebook will inherit values as global variables, as well as the authentication token you need to access the FOLIO API.

In [2]:
# --- EDIT THESE THREE VALUES ---
OKAPI_URL   = "https://api-your-institution.folio.ebsco.com"    # Your FOLIO API gateway URL (no trailing slash)
TENANT      = "fs0000000"                                       # Your FOLIO tenant ID
USERNAME    = "your-username"                                   # Your FOLIO username

# --- Password: Colab vs. local Jupyter ---
if IN_COLAB:
    # On Colab, secrets live in the notebook's "Secrets" panel (click the
    # key icon in the left sidebar), NOT as a shell environment variable --
    # Colab's cloud VM is ephemeral and has no persistent shell session for
    # `export` to write into. userdata.get() reads a secret you've stored
    # there by name.
    #
    # userdata.get() raises an error in two cases: the secret hasn't been
    # created yet, or it exists but this notebook hasn't been granted
    # access to it (each secret has a per-notebook access toggle in the
    # panel). We catch that broadly and just fall back to PASSWORD = None,
    # which later triggers the same getpass prompt used locally.
    try:
        PASSWORD = userdata.get("FOLIO_PASSWORD")
    except Exception:
        PASSWORD = None
    if PASSWORD is None:
        print("Tip: add a Colab secret named FOLIO_PASSWORD (key icon in the left "
              "sidebar) to avoid re-typing your password every session.")
else:
    # Local Jupyter: read from a real OS environment variable, set before
    # launching Jupyter (see instructions below).
    PASSWORD = os.environ.get("FOLIO_PASSWORD")

# Either way -- Colab or local -- if PASSWORD is still None at this point,
# folio_login() falls back to a hidden getpass prompt further down. So even
# if you skip Secrets/env vars entirely, the notebook still works; it just
# asks you to type your password each time.
#
# How to set FOLIO_PASSWORD as an environment variable before launching Jupyter (local only):
#
#   macOS / Linux (bash or zsh):
#     export FOLIO_PASSWORD='your-password-here'
#     (add this line to ~/.bashrc or ~/.zshrc to make it persist across sessions)
#
#   Windows (PowerShell):
#     $env:FOLIO_PASSWORD = 'your-password-here'        # current session only
#     setx FOLIO_PASSWORD "your-password-here"          # persists, but requires opening a NEW terminal
#
#   Windows (Command Prompt):
#     set FOLIO_PASSWORD=your-password-here             # current session only
#
# (Colab note: the env-var instructions above don't apply on Colab -- use the
# Secrets panel instead, as described above.)

In [3]:
class FolioAuthError(Exception):
    """Raised when FOLIO login fails or the expected token cookie is missing."""
    pass


def folio_login(okapi_url, tenant, username, password=None):
    """
    Log in to FOLIO via the cookie-based /authn/login-with-expiry endpoint.

    Returns (session, token):
        session : requests.Session with the auth cookie attached and
                  X-Okapi-Tenant / X-Okapi-Token set as default headers
        token   : the raw access token string

    Raises FolioAuthError if login fails or no folioAccessToken cookie
    is found in the response.
    """
    if password is None:
        password = getpass.getpass(f"FOLIO password for {username}: ")

    session = requests.Session()

    login_url = f"{okapi_url}/authn/login-with-expiry"
    headers = {
        "X-Okapi-Tenant": tenant,
        "Content-Type": "application/json",
    }
    payload = {
        "username": username,
        "password": password,
    }

    response = session.post(login_url, headers=headers, json=payload)

    if response.status_code not in (200, 201):
        raise FolioAuthError(
            f"Login failed with status {response.status_code}: {response.text}"
        )

    token = session.cookies.get("folioAccessToken")

    if not token:
        raise FolioAuthError(
            "Login returned a success status but no 'folioAccessToken' cookie "
            f"was found. Cookies received: {session.cookies.get_dict()}. "
            "Your tenant may use a different cookie name -- check with your "
            "FOLIO admin or the response above."
        )

    session.headers.update({
        "X-Okapi-Tenant": tenant,
        "X-Okapi-Token": token,
    })

    return session, token

## Log In

The function below is run automatically whenever this notebook is called from elsewhere. 

In [ ]:
try:
    session, token = folio_login(
        okapi_url=OKAPI_URL,
        tenant=TENANT,
        username=USERNAME,
        password=PASSWORD
    )
    print("Login succeeded. Token retrieved.")

except FolioAuthError as e:
    session, token = None, None
    print(f"Login failed: {e}")

Login succeeded. Token retrieved.
